# NB01 — Data Pipeline

**GraphSentry: A Unified GNN Framework for Blockchain Illicit Activity Detection**

This notebook constructs the complete data pipeline for GraphSentry. It loads the
Elliptic2 dataset, builds the background graph, extracts the 43 anonymised financial
features, performs a stratified train/validation/test split, and persists all
artefacts for downstream notebooks.

**Artefacts produced:**
- `background_graph.pt` — edge index, adjacency list, node/CC mappings
- `cc_metadata.pt` — labels, split indices, CC-level statistics
- `features_anonymous.pt` — normalised 43-dim financial features
- `scaler_anonymous.pkl` — fitted StandardScaler
- `dataset_stats.json` — summary statistics for reporting

**Environment:** Google Colab (T4 GPU, 16GB VRAM, ~12GB system RAM)

## 1. Configuration

All tuneable parameters live here. Change `SUBSET_MODE` to `False` for full-scale
training on ~122K subgraphs. Every downstream notebook loads the saved artefacts,
so the subset/full decision propagates automatically.

In [32]:
SUBSET_MODE = True
SUBSET_N_ILLICIT = 500
SUBSET_N_LICIT = 5_000

MIN_CC_SIZE = 5

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

CHUNK_SIZE = 500_000
N_ANONYMOUS_FEATURES = 43
SEED = 42

BASE_PATH = '/content/drive/MyDrive/GraphSentry'
RAW_PATH = f'{BASE_PATH}/data/raw'
PROCESSED_PATH = f'{BASE_PATH}/data/processed'

FEATURE_FILE = f'{RAW_PATH}/background_nodes.csv'
LABELS_FILE = f'{RAW_PATH}/connected_components.csv'
NODES_FILE = f'{RAW_PATH}/nodes.csv'
EDGES_FILE = f'{RAW_PATH}/edges.csv'

## 2. Environment setup

In [33]:
!pip install -q uv
!uv pip install --system torch torch-geometric numpy pandas tqdm scikit-learn

from google.colab import drive
import os, json, pickle, time, warnings

import torch
import numpy as np
import pandas as pd
from torch_geometric.data import Data
from torch_geometric.utils import degree
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm

warnings.filterwarnings('ignore')
np.random.seed(SEED)
torch.manual_seed(SEED)

drive.mount('/content/drive')
os.makedirs(PROCESSED_PATH, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Mode:   {'SUBSET (' + str(SUBSET_N_ILLICIT + SUBSET_N_LICIT) + ' CCs)' if SUBSET_MODE else 'FULL SCALE'}")

Using Python 3.12.13 environment at: /usr
Checked 6 packages in 93ms
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Mode:   SUBSET (5500 CCs)


## 3. Load raw data

Read the three small CSVs (labels, node membership, edges). The large feature file
(`background_nodes.csv`, ~77GB) is handled separately via chunked reading in Cell 6.

In [34]:
t0 = time.time()

df_labels = pd.read_csv(LABELS_FILE)
df_labels['label'] = (df_labels['ccLabel'] != 'licit').astype(int)

total_illicit = df_labels['label'].sum()
total_licit = len(df_labels) - total_illicit
print(f"Elliptic2 CCs: {len(df_labels):,} total ({total_illicit:,} illicit, {total_licit:,} licit)")

df_nodes_raw = pd.read_csv(NODES_FILE)
print(f"Elliptic2 nodes: {df_nodes_raw['clId'].nunique():,}")

df_edges_raw = pd.read_csv(EDGES_FILE)
print(f"Elliptic2 edges: {len(df_edges_raw):,}")

print(f"\nCSVs loaded in {time.time() - t0:.1f}s")

Elliptic2 CCs: 121,810 total (2,763 illicit, 119,047 licit)
Elliptic2 nodes: 444,521
Elliptic2 edges: 367,137

CSVs loaded in 0.3s


## 4. CC selection

Filter to CCs with at least `MIN_CC_SIZE` nodes (required for meaningful graph
structure). In subset mode, apply stratified sampling. In full-scale mode, use all
qualifying labelled CCs.

In [35]:
cc_sizes_df = df_nodes_raw.groupby('ccId').size().reset_index(name='size')
df_labels_with_size = df_labels.merge(cc_sizes_df, on='ccId', how='left')
df_labels_with_size = df_labels_with_size[df_labels_with_size['size'] >= MIN_CC_SIZE]

if SUBSET_MODE:
    illicit_ccs = df_labels_with_size[df_labels_with_size['label'] == 1].sample(
        n=min(SUBSET_N_ILLICIT, len(df_labels_with_size[df_labels_with_size['label'] == 1])),
        random_state=SEED
    )
    licit_ccs = df_labels_with_size[df_labels_with_size['label'] == 0].sample(
        n=min(SUBSET_N_LICIT, len(df_labels_with_size[df_labels_with_size['label'] == 0])),
        random_state=SEED
    )
    selected_ccs = pd.concat([illicit_ccs, licit_ccs]).sample(frac=1, random_state=SEED)
else:
    selected_ccs = df_labels_with_size[df_labels_with_size['label'].notna()].copy()

target_cc_ids = set(selected_ccs['ccId'].values)
cc_label_map = dict(zip(selected_ccs['ccId'], selected_ccs['label']))

n_illicit = selected_ccs['label'].sum()
n_licit = len(selected_ccs) - n_illicit
print(f"CCs with >= {MIN_CC_SIZE} nodes: {len(df_labels_with_size):,} "
      f"({df_labels_with_size['label'].sum():,} illicit, "
      f"{len(df_labels_with_size) - df_labels_with_size['label'].sum():,} licit)")
print(f"Selected CCs: {len(selected_ccs):,} ({n_illicit:,} illicit, {n_licit:,} licit)")
print(f"Class ratio: 1:{n_licit / max(n_illicit, 1):.1f} (illicit:licit)")

CCs with >= 5 nodes: 29,287 (788 illicit, 28,499 licit)
Selected CCs: 5,500 (500 illicit, 5,000 licit)
Class ratio: 1:10.0 (illicit:licit)


## 5. Build background graph

Construct the background graph restricted to selected CCs. This produces:
- A global node index mapping (`node_to_idx`)
- The CC membership mappings (`node_to_cc`, `cc_to_nodes`)
- The edge index as a `[2, M]` tensor
- An adjacency list for neighbourhood lookups and GraphSAINT random walk sampling

In [36]:
t0 = time.time()

df_nodes = df_nodes_raw[df_nodes_raw['ccId'].isin(target_cc_ids)].copy()
df_nodes = df_nodes.merge(selected_ccs[['ccId', 'label']], on='ccId')

unique_nodes = np.sort(df_nodes['clId'].unique())
node_to_idx = {int(nid): i for i, nid in enumerate(unique_nodes)}
N = len(unique_nodes)

node_to_cc = {}
cc_to_nodes = {}

for cc_id, group in df_nodes.groupby('ccId'):
    node_indices = [node_to_idx[nid] for nid in group['clId'].values]
    cc_to_nodes[cc_id] = sorted(node_indices)
    for idx in node_indices:
        node_to_cc[idx] = cc_id

node_id_set = set(unique_nodes)
edge_mask = df_edges_raw['clId1'].isin(node_id_set) & df_edges_raw['clId2'].isin(node_id_set)
df_edges = df_edges_raw[edge_mask]

src = np.array([node_to_idx[n] for n in df_edges['clId1'].values])
dst = np.array([node_to_idx[n] for n in df_edges['clId2'].values])
bg_edge_index = torch.tensor(np.stack([src, dst]), dtype=torch.long)

adj_list = [[] for _ in range(N)]
for s, d in zip(src, dst):
    adj_list[s].append(d)
    adj_list[d].append(s)

M = bg_edge_index.size(1)
cc_sizes = [len(nodes) for nodes in cc_to_nodes.values()]

print(f"Background graph constructed in {time.time() - t0:.1f}s")
print(f"  Nodes: {N:,}")
print(f"  Edges: {M:,}")
print(f"  CCs:   {len(cc_to_nodes):,}")
print(f"  CC sizes: min={min(cc_sizes)}, max={max(cc_sizes)}, "
      f"mean={np.mean(cc_sizes):.1f}, median={np.median(cc_sizes):.0f}")

Background graph constructed in 0.4s
  Nodes: 39,819
  Edges: 38,037
  CCs:   5,500
  CC sizes: min=5, max=92, mean=7.2, median=6


## 6. Feature extraction (43 dims)

Load the 43 anonymised financial features from `background_nodes.csv` using chunked
reading to manage the ~77GB file size. Normalisation happens after the split in
Cell 7 to prevent data leakage.

In [37]:
t0 = time.time()
features_anonymous = torch.zeros((N, N_ANONYMOUS_FEATURES), dtype=torch.float)
nodes_found = 0

chunk_iter = pd.read_csv(FEATURE_FILE, chunksize=CHUNK_SIZE)
for chunk in tqdm(chunk_iter, desc="Loading features"):
    mask = chunk['clId'].isin(node_id_set)
    matched = chunk[mask]
    if len(matched) == 0:
        continue

    feat_cols = [c for c in matched.columns if c.startswith('feat')]
    for _, row in matched.iterrows():
        idx = node_to_idx.get(row['clId'])
        if idx is not None:
            features_anonymous[idx] = torch.tensor(
                row[feat_cols].values.astype(np.float32)
            )
            nodes_found += 1

coverage = nodes_found / N * 100
print(f"Features loaded in {time.time() - t0:.1f}s")
print(f"  Shape: {features_anonymous.shape}")
print(f"  Coverage: {nodes_found:,} / {N:,} nodes ({coverage:.1f}%)")
print(f"  Non-zero rows: {(features_anonymous != 0).any(dim=1).sum().item():,}")

Loading features: 99it [02:32,  1.54s/it]

Features loaded in 153.0s
  Shape: torch.Size([39819, 43])
  Coverage: 39,819 / 39,819 nodes (100.0%)
  Non-zero rows: 39,819


## 7. Train / validation / test split

Split at the CC level with stratification by label. The validation set is used for
early stopping and hyperparameter selection. The test set is touched only once for
final reported metrics.

After splitting, fit the StandardScaler on training data only to prevent data leakage.

In [38]:
all_cc_ids = np.array(sorted(cc_to_nodes.keys()))
all_labels = np.array([cc_label_map[cc] for cc in all_cc_ids])

# train vs (val + test)
train_ids, temp_ids, train_labels, temp_labels = train_test_split(
    all_cc_ids, all_labels,
    test_size=(VAL_RATIO + TEST_RATIO),
    stratify=all_labels,
    random_state=SEED,
)

# val vs test, proportional within the temp set
relative_test = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
val_ids, test_ids, val_labels, test_labels = train_test_split(
    temp_ids, temp_labels,
    test_size=relative_test,
    stratify=temp_labels,
    random_state=SEED,
)

split_map = {}
for cc_id in train_ids:
    split_map[cc_id] = 'train'
for cc_id in val_ids:
    split_map[cc_id] = 'val'
for cc_id in test_ids:
    split_map[cc_id] = 'test'

def _split_summary(ids, labels, name):
    n_ill = labels.sum()
    return f"  {name:6s}: {len(ids):>6,} CCs ({n_ill:,} illicit, {len(ids) - n_ill:,} licit)"

print("Split summary:")
print(_split_summary(train_ids, train_labels, "Train"))
print(_split_summary(val_ids, val_labels, "Val"))
print(_split_summary(test_ids, test_labels, "Test"))

train_node_indices = []
for cc_id in train_ids:
    train_node_indices.extend(cc_to_nodes[cc_id])
train_node_indices = np.array(train_node_indices)

anon_train = features_anonymous[train_node_indices].numpy()
valid_mask = (anon_train != 0).any(axis=1)
scaler_anonymous = StandardScaler()
scaler_anonymous.fit(anon_train[valid_mask])

features_anonymous_norm = torch.tensor(
    scaler_anonymous.transform(features_anonymous.numpy()), dtype=torch.float
)

print(f"\nScaler fitted on {len(train_node_indices):,} training nodes ({valid_mask.sum():,} valid rows)")

Split summary:
  Train :  3,850 CCs (350 illicit, 3,500 licit)
  Val   :    825 CCs (75 illicit, 750 licit)
  Test  :    825 CCs (75 illicit, 750 licit)

Scaler fitted on 27,879 training nodes (27,879 valid rows)


## 8. Save artefacts

Persist everything downstream notebooks need. Each artefact is self-contained;
no downstream notebook reads the raw CSVs.

In [39]:
t0 = time.time()

cc_ids_ordered = sorted(cc_to_nodes.keys())

torch.save({
    'edge_index': bg_edge_index,
    'adj_list': adj_list,
    'node_to_idx': node_to_idx,
    'node_to_cc': node_to_cc,
    'cc_to_nodes': cc_to_nodes,
    'num_nodes': N,
    'num_edges': M,
}, os.path.join(PROCESSED_PATH, 'background_graph.pt'))

torch.save({
    'cc_label_map': cc_label_map,
    'split_map': split_map,
    'train_ids': train_ids.tolist(),
    'val_ids': val_ids.tolist(),
    'test_ids': test_ids.tolist(),
    'cc_ids_ordered': [int(x) for x in cc_ids_ordered],
}, os.path.join(PROCESSED_PATH, 'cc_metadata.pt'))

torch.save(features_anonymous_norm, os.path.join(PROCESSED_PATH, 'features_anonymous.pt'))

with open(os.path.join(PROCESSED_PATH, 'scaler_anonymous.pkl'), 'wb') as f:
    pickle.dump(scaler_anonymous, f)

stats = {
    'subset_mode': SUBSET_MODE,
    'seed': SEED,
    'min_cc_size': MIN_CC_SIZE,
    'total_ccs': len(cc_to_nodes),
    'total_nodes': N,
    'total_edges': M,
    'n_illicit': int(n_illicit),
    'n_licit': int(n_licit),
    'class_ratio': round(n_licit / max(n_illicit, 1), 1),
    'split': {
        'train': len(train_ids),
        'val': len(val_ids),
        'test': len(test_ids),
    },
    'split_illicit': {
        'train': int(train_labels.sum()),
        'val': int(val_labels.sum()),
        'test': int(test_labels.sum()),
    },
    'cc_sizes': {
        'min': int(min(cc_sizes)),
        'max': int(max(cc_sizes)),
        'mean': round(float(np.mean(cc_sizes)), 1),
        'median': int(np.median(cc_sizes)),
    },
    'feature_dims': {
        'anonymous': N_ANONYMOUS_FEATURES,
    },
}

with open(os.path.join(PROCESSED_PATH, 'dataset_stats.json'), 'w') as f:
    json.dump(stats, f, indent=2)

print(f"Artefacts saved to {PROCESSED_PATH} in {time.time() - t0:.1f}s\n")
for fname in sorted(os.listdir(PROCESSED_PATH)):
    fpath = os.path.join(PROCESSED_PATH, fname)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {fname:<30s} {size_mb:>8.2f} MB")

Artefacts saved to /content/drive/MyDrive/GraphSentry/data/processed in 0.7s

  background_graph.pt                5.25 MB
  cc_metadata.pt                     0.34 MB
  dataset_stats.json                 0.00 MB
  demo_data.pt                       0.14 MB
  features_anonymous.pt              6.53 MB
  features_topology.pt               1.22 MB
  final_mvp.pth                      0.16 MB
  model_b.pth                        0.14 MB
  model_weights.pt                   0.16 MB
  mvp_dataset.pt                    17.29 MB
  scaler_anonymous.pkl               0.00 MB
  scaler_topology.pkl                0.00 MB
  training_results.json              0.00 MB


## 9. Validation

Reload all artefacts from disk and verify shapes, coverage, and consistency.

In [40]:
print("=" * 60)
print("VALIDATION: Reloading artefacts from disk")
print("=" * 60)

bg = torch.load(os.path.join(PROCESSED_PATH, 'background_graph.pt'), weights_only=False)
meta = torch.load(os.path.join(PROCESSED_PATH, 'cc_metadata.pt'), weights_only=False)
feat_anon = torch.load(os.path.join(PROCESSED_PATH, 'features_anonymous.pt'), weights_only=False)

with open(os.path.join(PROCESSED_PATH, 'scaler_anonymous.pkl'), 'rb') as f:
    sc_anon = pickle.load(f)
with open(os.path.join(PROCESSED_PATH, 'dataset_stats.json'), 'r') as f:
    loaded_stats = json.load(f)

assert bg['edge_index'].shape[0] == 2
assert feat_anon.shape == (bg['num_nodes'], N_ANONYMOUS_FEATURES)
assert len(bg['adj_list']) == bg['num_nodes']

all_split_ids = set(meta['train_ids'] + meta['val_ids'] + meta['test_ids'])
assert len(all_split_ids) == len(bg['cc_to_nodes'])

train_set = set(meta['train_ids'])
val_set = set(meta['val_ids'])
test_set = set(meta['test_ids'])
assert len(train_set & val_set) == 0
assert len(train_set & test_set) == 0
assert len(val_set & test_set) == 0

assert sc_anon.n_features_in_ == N_ANONYMOUS_FEATURES

train_nodes = []
for cc_id in meta['train_ids']:
    train_nodes.extend(bg['cc_to_nodes'][cc_id])
train_mean = feat_anon[train_nodes].mean(dim=0).abs().max().item()
assert train_mean < 0.5, f"Features poorly normalised (max mean = {train_mean:.3f})"

print("\nAll checks passed.\n")
print(json.dumps(loaded_stats, indent=2))

VALIDATION: Reloading artefacts from disk

All checks passed.

{
  "subset_mode": true,
  "seed": 42,
  "min_cc_size": 5,
  "total_ccs": 5500,
  "total_nodes": 39819,
  "total_edges": 38037,
  "n_illicit": 500,
  "n_licit": 5000,
  "class_ratio": 10.0,
  "split": {
    "train": 3850,
    "val": 825,
    "test": 825
  },
  "split_illicit": {
    "train": 350,
    "val": 75,
    "test": 75
  },
  "cc_sizes": {
    "min": 5,
    "max": 92,
    "mean": 7.2,
    "median": 6
  },
  "feature_dims": {
    "anonymous": 43
  }
}


## 10. Dataset preview

Build sample PyG Data objects to verify features + edges + labels produce valid graphs.

In [41]:
def build_pyg_graph(cc_id, features, bg_data):
    """
    Build a PyG Data object for a single CC with an appended degree feature.
    Reused in all downstream notebooks.
    """
    cc_nodes = sorted(bg_data['cc_to_nodes'][cc_id])
    local_map = {g: l for l, g in enumerate(cc_nodes)}
    adj = bg_data['adj_list']

    x = features[cc_nodes].clone()

    node_set = set(cc_nodes)
    local_src, local_dst = [], []
    for g_src in cc_nodes:
        for g_dst in adj[g_src]:
            if g_dst in node_set and g_dst in local_map:
                local_src.append(local_map[g_src])
                local_dst.append(local_map[g_dst])

    if len(local_src) == 0:
        edge_index = torch.tensor([[0], [0]], dtype=torch.long)
    else:
        edge_index = torch.tensor([local_src, local_dst], dtype=torch.long)

    deg = degree(edge_index[1], x.size(0), dtype=torch.float)
    deg = torch.log(deg + 1).view(-1, 1)
    x = torch.cat([x, deg], dim=1)

    y = torch.tensor([meta['cc_label_map'][cc_id]], dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=y)


print(f"{'CC ID':>10s}  {'Split':>6s}  {'Label':>6s}  {'Nodes':>6s}  {'Edges':>6s}  {'Features':>10s}")
print("-" * 60)

for split_name, split_ids in [('train', meta['train_ids'][:3]),
                               ('val', meta['val_ids'][:1]),
                               ('test', meta['test_ids'][:1])]:
    for cc_id in split_ids:
        g = build_pyg_graph(cc_id, feat_anon, bg)
        label = 'illicit' if g.y.item() == 1 else 'licit'
        print(f"{cc_id:>10d}  {split_name:>6s}  {label:>6s}  "
              f"{g.num_nodes:>6d}  {g.num_edges:>6d}  "
              f"{str(list(g.x.shape)):>10s}")

print("\nPipeline validated. Ready for NB02.")

     CC ID   Split   Label   Nodes   Edges    Features
------------------------------------------------------------
      8381   train  illicit       7      12     [7, 44]
      9950   train   licit       7      14     [7, 44]
      1670   train   licit      13      26    [13, 44]
      3923     val   licit      10      20    [10, 44]
     16570    test   licit       6      12     [6, 44]

Pipeline validated. Ready for NB02.
